In [1]:
import torch

print(f"PyTorch 版本: {torch.__version__}")

print(f"CUDA 可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():

    print(f"GPU 设备: {torch.cuda.get_device_name(0)}")

    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch 版本: 2.4.1+cu121
CUDA 可用: True
GPU 设备: Tesla V100-SXM2-32GB
GPU 显存: 34.08 GB


In [2]:
import os

import json
import pandas as pd
from datetime import datetime
from tqdm import tqdm
tqdm.pandas()  # 启用 pandas progress_apply

import asyncio
import httpx
import uuid
import nest_asyncio
nest_asyncio.apply()

from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, AutoPeftModelForCausalLM
from trl import SFTTrainer
from datasets import Dataset

print("✅ 所有库导入成功!")

✅ 所有库导入成功!


In [3]:
model_path = "/home/sankuai/dolphinfs_chixinning/huggingface.co/microsoft/phi-1_5"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

In [4]:
!nvidia-smi


Mon Apr 27 14:01:36 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla V100-SXM2-32GB           On  | 00000000:3D:00.0 Off |                    0 |
| N/A   44C    P0              43W / 300W |      3MiB / 32768MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [5]:
CONFIG = {
    # API 配置
    "api_key": os.getenv("LLM_API_KEY", "22046202597199609929"),  # 请替换为你的 API Key
    "api_url": "https://aigc.sankuai.com/v1/openai/native/chat/completions",
    "distill_model": "LongCat-8B-128K-Chat",
    "request_delay": 3.0,  # API 请求间隔(秒)

    # 模型配置
    "base_model": f"/home/sankuai/dolphinfs_chixinning/huggingface.co/Qwen/Qwen2.5-7B-Instruct",  # 本地路径或 HuggingFace ID
    "output_dir": "./sft_output_gpu_Qwen",

    # LoRA 配置
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.1,

    # 训练配置
    "batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 1e-5,
    "num_epochs": 3,
    "max_seq_length": 2048,

    # 数据配置
    "input_csv": "test_SFT_demo.csv",  # 你的输入数据文件
    "input_excel": "SFT demo数据取数逻辑0423_cut_2_22875397_1776912256.xlsx",
    "distill_limit": 50,  # 测试时可设置为 10

}
print("📋 当前配置:")

for key, value in CONFIG.items():
    print(f"  • {key}: {value}")

📋 当前配置:
  • api_key: 22046202597199609929
  • api_url: https://aigc.sankuai.com/v1/openai/native/chat/completions
  • distill_model: LongCat-8B-128K-Chat
  • request_delay: 3.0
  • base_model: /home/sankuai/dolphinfs_chixinning/huggingface.co/Qwen/Qwen2.5-7B-Instruct
  • output_dir: ./sft_output_gpu_Qwen
  • lora_r: 8
  • lora_alpha: 16
  • lora_dropout: 0.1
  • batch_size: 2
  • gradient_accumulation_steps: 8
  • learning_rate: 1e-05
  • num_epochs: 3
  • max_seq_length: 2048
  • input_csv: test_SFT_demo.csv
  • input_excel: SFT demo数据取数逻辑0423_cut_2_22875397_1776912256.xlsx
  • distill_limit: 50


In [6]:
if os.path.exists(CONFIG["input_csv"]):
    df_raw = pd.read_excel(CONFIG["input_excel"])
    df_raw['apply_m'] = df_raw['open_date'].str[:7]
    for y_col in [
        'dpd30_n3_loan',
        'dpd15_n2_loan',
        'dpd7_n1_loan' ,
    ]:
        df_raw[y_col]=1-abs(df_raw[y_col])

    print(f"✅ 成功加载数据: {len(df_raw)} 条")
    print(f"\n📊 数据列: {df_raw.columns.tolist()}")
    print(f"\n前3行预览:")
    display(df_raw.head(3))

else:
    print(f"❌ 文件不存在: {CONFIG['input_csv']}")
    raise Error


    _ =df_raw.groupby('apply_m').agg({
        'dpd30_n3_loan':['mean','sum','count'],
        'dpd15_n2_loan':['mean','sum','count'],
        'dpd7_n1_loan':['mean','sum','count'],
    })
    display(_)

✅ 成功加载数据: 221 条

📊 数据列: ['loan_no', 'user_id', 'customer_no', 'open_date', 'join_date', 'rnk', 'first_open_date', 'is_everloan', 'first_loan_date', 'mt_bal', 'mt_limit', 'mt_rate', 'overdue_cur_max_days', 'overdue_his_max_days', 'account_info', 'credit_apply_info', 'cus_close_info', 'limit_info', 'rate_info_01', 'rate_info_27', 'rate_info_17', 'loan_info', 'repay_info', 'cs_online_info', 'cs_info', 'collect_info', 'pbc_info', 'couple_info', 'recent_address_info', 'recent_app_info', 'mot_3m_info', 'user_base_fact_info', 'order_address_info', 'geohash_address_info', 'wifi_address_info', 'pboc_info', 'order_total_info', 'university_info', 'cur_app_info', 'his_app_info', 'wifi_info_5years', 'poi_info_5years', 'address_info_5years', 'poi_info_year_top10', 'poi_info_3m', 'order_detail_info', 'month_consume_info', 'refund_info_p3m', 'plane_and_train_info', 'pay_fail_order_info', 'buy_but_nocons_order_info', 'rider_info', 'search_word_info', 'pboc_input', 'benren_couple_info', 'user_debt_info'

,loan_no,user_id,customer_no,open_date,join_date,rnk,first_open_date,is_everloan,first_loan_date,mt_bal,...,is_common_approve_pass,credit_loan_diff_days,is_loan_success,is_policy_pass,dpd30_n3_loan,dpd15_n3_loan,dpd30_n1_loan,dpd15_n2_loan,dpd7_n1_loan,apply_m
0,1975012908526985256,2570951852,1000000314902102,2025-10-06,2025-08-31,77036,NaN,NaN,NaN,NaN,...,1,0,1,1,0,1,1,0,0,2025-10
1,1976527262941896735,1806148074,1000000356130978,2025-10-10,2025-08-31,72378,NaN,NaN,NaN,NaN,...,1,11,1,1,0,1,1,0,0,2025-10
2,1982744173300703255,1788169156,1000000327073983,2025-10-27,2025-08-31,79940,2024-07-18,0.0,NaN,NaN,...,1,466,1,1,0,1,1,0,0,2025-10


In [7]:
from tqdm import tqdm
tqdm.pandas() 

In [8]:
def get_input_text(row):
    return f'''【任务指令】
你现在是美团信贷风控部门的资深风险分析师。请根据以下提供的多维度客户信息，运用你的专业风控知识，对该客户的信用风险进行深度剖析，并最终预测其在美团借款后是否会发生逾期行为。
【分析要求】
请综合下方数据，识别客户是否存在多头借贷、还款意愿下降、收入不稳定性或异常消费行为等风险特征，并给出该客户是否会逾期的判断。
遵循的基本规则：
    1. 评估信贷厚度与真实履约能力：对于“白户”或信贷记录极少的用户，历史无逾期仅代表“未暴露风险”，不等于“信用良好”；需结合其收入、资产及消费稳定性综合判断。
    2. 区分债务压力与授信敞口：准确区分“已用负债”（还款压力）与“可用额度/供给”（抗风险拆借能力）。重点关注【额度使用率】，若多产品额度使用率极高且接近满额，通常预示资金链紧张。
    3. 警惕近期信用恶化（近因效应）：风险判断需赋予近期行为更高权重。近期（近1-3个月）出现的频繁多头查询、新增多头借贷、历史最大逾期天数增加，是强烈的违约前兆。
    4. 关注行为突变与异常：捕捉用户在借贷、支付、消费、LBS轨迹及APP偏好上的“断崖式变化”。例如：突然密集安装各类网贷APP、消费水平骤降、频繁支付失败或退款、常驻地址突变等，均指向风险恶化。
    5. 识别关系网传染风险：配偶的微贷逾期行为、用户本人的历史催收进线记录，具有极强的风险传染性，需作为高危特征对待。
    6. 画像交叉验证：验证用户的消费水平、重大支出线索是否与其负债规模、职业身份相匹配。防范过度消费导致的还款意愿与能力双降。

【客户多维数据】
1. 基本身份信息：
   - 基础画像：{row['user_base_fact_info']}
   - 教育背景：{row['university_info']}

2. 信用与负债表现：
   - 官方征信概况：{row['pboc_info']}
   - 近三年人行明细：{row['pbc_info']}
   - 整体负债压力：{row['user_debt_info']}

3. 美团内部账户表现（四产品：生活费小额01/中额27、生意贷小额07/中额17）：
   - 开户与首借：最早开立于{row['first_open_date']}，首借日期{row['first_loan_date']}，是否曾借款：{row['is_everloan']}。
   - 额度与定价：总余额{row['mt_bal']}，总额度{row['mt_limit']}，最低定价{row['mt_rate']}。
   - 逾期历史：当前最大逾期{row['overdue_cur_max_days']}天，历史最大逾期{row['overdue_his_max_days']}天。
   - 动态信息：当前授信{row['account_info']}。历史申请{row['credit_apply_info']}。关户记录{row['cus_close_info']}。调额记录{row['limit_info']}。调价记录(01/27/17)：{row['rate_info_01']}/{row['rate_info_27']}/{row['rate_info_17']}。
   - 借还款明细：近三年申请与明细{row['loan_info']}。近三年额度使用与还款{row['repay_info']}。

4. 交互与服务记录：
   - 客服记录：线上咨询{row['cs_online_info']}。进线沟通{row['cs_info']}。
   - 催收记录：历史催收沟通情况{row['collect_info']}。

5. 社交与家庭关联：
   - 配偶概况：{row['benren_couple_info']}；配偶微贷行为：{row['couple_info']}。

6. 生活行为与消费偏好：
   - 地理位置：常用收货地址{row['order_address_info']}。打点地址{row['geohash_address_info']}；常用WIFI地址{row['wifi_address_info']}。
   - 消费能力：月度消费汇总{row['month_consume_info']}。近期重大支出线索{row['mot_3m_info']}；订单明细{row['order_total_info']}。商旅支出{row['plane_and_train_info']}。
   - 交易稳定性：退款记录{row['refund_info_p3m']}。支付失败记录{row['pay_fail_order_info']}。
   - 兴趣与偏好：在装APP{row['cur_app_info']}。历史APP安装{row['his_app_info']}。搜索关键词{row['search_word_info']}。

'''

df_raw['text'] = df_raw.progress_apply(get_input_text, axis=1)

100%|██████████| 221/221 [00:00<00:00, 8980.33it/s]


In [9]:
print(df_raw['text'].values[0])


【任务指令】
你现在是美团信贷风控部门的资深风险分析师。请根据以下提供的多维度客户信息，运用你的专业风控知识，对该客户的信用风险进行深度剖析，并最终预测其在美团借款后是否会发生逾期行为。
【分析要求】
请综合下方数据，识别客户是否存在多头借贷、还款意愿下降、收入不稳定性或异常消费行为等风险特征，并给出该客户是否会逾期的判断。
遵循的基本规则：
    1. 评估信贷厚度与真实履约能力：对于“白户”或信贷记录极少的用户，历史无逾期仅代表“未暴露风险”，不等于“信用良好”；需结合其收入、资产及消费稳定性综合判断。
    2. 区分债务压力与授信敞口：准确区分“已用负债”（还款压力）与“可用额度/供给”（抗风险拆借能力）。重点关注【额度使用率】，若多产品额度使用率极高且接近满额，通常预示资金链紧张。
    3. 警惕近期信用恶化（近因效应）：风险判断需赋予近期行为更高权重。近期（近1-3个月）出现的频繁多头查询、新增多头借贷、历史最大逾期天数增加，是强烈的违约前兆。
    4. 关注行为突变与异常：捕捉用户在借贷、支付、消费、LBS轨迹及APP偏好上的“断崖式变化”。例如：突然密集安装各类网贷APP、消费水平骤降、频繁支付失败或退款、常驻地址突变等，均指向风险恶化。
    5. 识别关系网传染风险：配偶的微贷逾期行为、用户本人的历史催收进线记录，具有极强的风险传染性，需作为高危特征对待。
    6. 画像交叉验证：验证用户的消费水平、重大支出线索是否与其负债规模、职业身份相匹配。防范过度消费导致的还款意愿与能力双降。

【客户多维数据】
1. 基本身份信息：
   - 基础画像：当前日期：2025-08-31。年龄：25。性别：女。常住地：河北省唐山市路北区。出生地：未知。户籍地：未知。户籍地址：未知。关联企业信息：。微信昵称：星星🌟。微信昵称获取日期：2023-07-26，微信昵称末次获取日期：2023-08-15，美团实名时间：2021-04-01 22:46:47，美团实名下userid数量：3
   - 教育背景：认证学校：河北东方学院，入学时间：2020-09，毕业时间：2024-06，一般可信

2. 信用与负债表现：
   - 官方征信概况：征信报告房贷借款日期：0。征信报告房贷借款金额：0。征信报告房贷月供：0。征信报告车贷借款日期：0。征信报告

In [10]:
def build_prompt_from_row(row):
    """
    根据数据行构造蒸馏 Prompt
    返回:
        - system_prompt: 系统提示词
        - user_input: 给小模型的输入
        - distill_prompt: 给大模型的完整输入(包含蒸馏指令)
    """
    # 提取标签字段
    dpd7_n1_loan = row.get('dpd7_n1_loan', 0) or 0
    dpd15_n2_loan = row.get('dpd15_n2_loan', 0) or 0
    dpd30_n3_loan = row.get('dpd30_n3_loan', 0) or 0
    
    # 判断风险等级
    has_high_risk = (dpd7_n1_loan > 0) or (dpd15_n2_loan > 0) or (dpd30_n3_loan > 0)
    ideal_decision = "REJECT" if has_high_risk else "APPROVE"

    # 系统提示词 - 与 get_input_text 保持一致的风控专家角色
    system_prompt = """你是美团信贷风控部门的资深风险分析师，专注于消费信贷领域的风险评估与客户画像分析。
你的核心任务是根据客户的多维度信息，识别客户是否存在多头借贷、还款意愿下降、收入不稳定性或异常消费行为等风险特征，并给出该客户是否会逾期的判断。

【遵循的基本规则】
1. 评估信贷厚度与真实履约能力：对于"白户"或信贷记录极少的用户，历史无逾期仅代表"未暴露风险"，不等于"信用良好"；需结合其收入、资产及消费稳定性综合判断。
2. 区分债务压力与授信敞口：准确区分"已用负债"（还款压力）与"可用额度/供给"（抗风险拆借能力）。重点关注【额度使用率】，若多产品额度使用率极高且接近满额，通常预示资金链紧张。
3. 警惕近期信用恶化（近因效应）：风险判断需赋予近期行为更高权重。近期（近1-3个月）出现的频繁多头查询、新增多头借贷、历史最大逾期天数增加，是强烈的违约前兆。
4. 关注行为突变与异常：捕捉用户在借贷、支付、消费、LBS轨迹及APP偏好上的"断崖式变化"。例如：突然密集安装各类网贷APP、消费水平骤降、频繁支付失败或退款、常驻地址突变等，均指向风险恶化。
5. 识别关系网传染风险：配偶的微贷逾期行为、用户本人的历史催收进线记录，具有极强的风险传染性，需作为高危特征对待。
6. 画像交叉验证：验证用户的消费水平、重大支出线索是否与其负债规模、职业身份相匹配。防范过度消费导致的还款意愿与能力双降。

【输出要求】
请输出严格的JSON格式，包含以下字段：
1. "customer_profile": 客户画像概述（不少于100字）
2. "risk_analysis": 风险分析（不少于80字，从负债结构、还款能力、逾期风险、多头查询等维度分析）
3. "demand_analysis": 需求分析（不少于50字，分析客户的用信需求和资金用途）
4. "reasoning_process": 决策推导过程（不少于80字的思维链推导）
5. "credit_score": 预估信用分（0-1000）
6. "decision": 最终决定（只能是APPROVE或REJECT）
7. "key_risk_factors": 关键风险因素列表（数组格式）
8. "suggestions": 风控建议"""

    # 用户输入 - 与 get_input_text 保持一致的数据结构
    user_input = f"""【任务指令】
请根据以下提供的多维度客户信息，对该客户的信用风险进行深度剖析，并最终预测其在美团借款后是否会发生逾期行为。

【客户多维数据】
1. 基本身份信息：
   - 基础画像：{row.get('user_base_fact_info', '无')}
   - 教育背景：{row.get('university_info', '无')}

2. 信用与负债表现：
   - 官方征信概况：{row.get('pboc_info', '无')}
   - 近三年人行明细：{row.get('pbc_info', '无')}
   - 整体负债压力：{row.get('user_debt_info', '无')}

3. 美团内部账户表现（四产品：生活费小额01/中额27、生意贷小额07/中额17）：
   - 开户与首借：最早开立于{row.get('first_open_date', '无')}，首借日期{row.get('first_loan_date', '无')}，是否曾借款：{row.get('is_everloan', '无')}。
   - 额度与定价：总余额{row.get('mt_bal', 0)}，总额度{row.get('mt_limit', 0)}，最低定价{row.get('mt_rate', '无')}。
   - 逾期历史：当前最大逾期{row.get('overdue_cur_max_days', 0)}天，历史最大逾期{row.get('overdue_his_max_days', 0)}天。
   - 动态信息：当前授信{row.get('account_info', '无')}。历史申请{row.get('credit_apply_info', '无')}。关户记录{row.get('cus_close_info', '无')}。调额记录{row.get('limit_info', '无')}。调价记录(01/27/17)：{row.get('rate_info_01', '无')}/{row.get('rate_info_27', '无')}/{row.get('rate_info_17', '无')}。
   - 借还款明细：近三年申请与明细{row.get('loan_info', '无')}。近三年额度使用与还款{row.get('repay_info', '无')}。

4. 交互与服务记录：
   - 客服记录：线上咨询{row.get('cs_online_info', '无')}。进线沟通{row.get('cs_info', '无')}。
   - 催收记录：历史催收沟通情况{row.get('collect_info', '无')}。

5. 社交与家庭关联：
   - 配偶概况：{row.get('benren_couple_info', '无')}；配偶微贷行为：{row.get('couple_info', '无')}。

6. 生活行为与消费偏好：
   - 地理位置：常用收货地址{row.get('order_address_info', '无')}。打点地址{row.get('geohash_address_info', '无')}；常用WIFI地址{row.get('wifi_address_info', '无')}。
   - 消费能力：月度消费汇总{row.get('month_consume_info', '无')}。近期重大支出线索{row.get('mot_3m_info', '无')}；订单明细{row.get('order_total_info', '无')}。商旅支出{row.get('plane_and_train_info', '无')}。
   - 交易稳定性：退款记录{row.get('refund_info_p3m', '无')}。支付失败记录{row.get('pay_fail_order_info', '无')}。
   - 兴趣与偏好：在装APP{row.get('cur_app_info', '无')}。历史APP安装{row.get('his_app_info', '无')}。搜索关键词{row.get('search_word_info', '无')}。
"""

    # 蒸馏指令 - 包含标签信息
    distill_instruction = f"""

【内部指令 - 仅用于模型训练】
该客户的真实表现已被验证：
- 目标决策: {ideal_decision}
- DPD7+逾期(近1期): {dpd7_n1_loan}
- DPD15+逾期(近2期): {dpd15_n2_loan}
- DPD30+逾期(近3期): {dpd30_n3_loan}

请基于上述目标决策，逆向推导出一套符合风控逻辑的判断理由，并按照JSON格式输出完整的分析结果。注意：
1. 如果目标是REJECT，需要从风险角度给出充分的拒绝理由
2. 如果目标是APPROVE，需要说明该客户的风险可控点和可接受原因
3. 分析过程需要遵循前述基本规则，确保逻辑严密"""

    return system_prompt, user_input, user_input + distill_instruction

In [11]:
async def call_api_async(system_prompt, user_prompt, api_key, api_url, model_name):
    """异步调用大模型 API"""
    trace_id = str(uuid.uuid4())
    
    async with httpx.AsyncClient(timeout=60.0) as client:
        response = await client.post(
            api_url,
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
                "M-TraceId": trace_id
            },
            json={
                "model": model_name,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                "temperature": 0.3,
                "max_tokens": 1000,
                "user": trace_id
            }
        )
        
        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"]
        else:
            raise Exception(f"API错误 {response.status_code}: {response.text[:200]}")


def parse_json_response(result_str):
    """解析 API 返回的 JSON"""
    json_str = result_str
    
    # 处理 markdown 代码块
    if "```json" in result_str:
        start = result_str.find("```json") + 7
        end = result_str.find("```", start)
        json_str = result_str[start:end].strip()
    elif "```" in result_str:
        start = result_str.find("```") + 3
        end = result_str.find("```", start)
        json_str = result_str[start:end].strip()
    elif "{" in result_str and "}" in result_str:
        start = result_str.find("{")
        end = result_str.rfind("}") + 1
        json_str = result_str[start:end]
    
    # 验证 JSON
    parsed = json.loads(json_str)
    return json_str, parsed

print("✅ API 函数定义完成")

✅ API 函数定义完成


In [12]:
# 输出文件
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_jsonl = f"distilled_data_{timestamp}.jsonl"
output_jsonl = "distilled_data_20260424_111123.jsonl"

# print(f"\n{'='*60}")
# print(f"🔨 开始数据蒸馏")
# print(f"{'='*60}")
# print(f"输出文件: {output_jsonl}")

# # 检查 API Key
# if CONFIG["api_key"] == "your-api-key":
#     print("\n❌ 请先配置 API Key!")
#     print("方法1: export LLM_API_KEY='你的AppID'")
#     print("方法2: 在上面的 CONFIG 字典中直接设置 api_key")
# else:
#     # 限制数据量(测试用)
#     df_process = df_raw.head(CONFIG["distill_limit"]) if CONFIG["distill_limit"] else df_raw
    
#     success_count = 0
    
#     with open(output_jsonl, "w", encoding="utf-8") as f:
#         for index, row in tqdm(df_process.iterrows(), total=len(df_process), desc="蒸馏进度"):
#             try:
#                 # 构造 Prompt
#                 system_prompt, user_input, distill_prompt = build_prompt_from_row(row)
                
#                 # 调用 API
#                 result_str = asyncio.run(call_api_async(
#                     system_prompt, 
#                     distill_prompt,
#                     CONFIG["api_key"],
#                     CONFIG["api_url"],
#                     CONFIG["distill_model"]
#                 ))
                
#                 # 解析 JSON
#                 json_str, parsed = parse_json_response(result_str)
                
#                 # 保存为 ShareGPT 格式
#                 sft_record = {
#                     "messages": [
#                         {"role": "system", "content": "你是一个专业的信贷风控决策引擎。你的任务是根据用户的输入特征，进行严谨的逻辑推理，并严格按照 JSON 格式输出决策结果。决不能编造数据。"},
#                         {"role": "user", "content": user_input},
#                         {"role": "assistant", "content": json_str}
#                     ]
#                 }
                
#                 f.write(json.dumps(sft_record, ensure_ascii=False) + "\n")
#                 success_count += 1
                
#             except Exception as e:
#                 print(f"\n⚠️ 第 {index} 条处理失败: {e}")
            
#             # 限流
#             import time
#             time.sleep(CONFIG["request_delay"])
    
#     print(f"\n✅ 数据蒸馏完成: {success_count}/{len(df_process)} 条成功")
#     print(f"📁 输出文件: {output_jsonl}")

In [13]:
# 读取并展示第一条数据
if os.path.exists(output_jsonl):
    with open(output_jsonl, 'r', encoding='utf-8') as f:
        first_line = json.loads(f.readline())
    
    print("📝 第一条蒸馏数据:")
    print("\n【System】:")
    print(first_line['messages'][0]['content'][:100] + "...")
    print("\n【User】:")
    print(first_line['messages'][1]['content'][:200] + "...")
    print("\n【Assistant】:")
    print(first_line['messages'][2]['content'])
else:
    print("⚠️ 蒸馏文件不存在，请先运行上面的蒸馏代码")

📝 第一条蒸馏数据:

【System】:
你是一个专业的信贷风控决策引擎。你的任务是根据用户的输入特征，进行严谨的逻辑推理，并严格按照 JSON 格式输出决策结果。决不能编造数据。...

【User】:
【任务指令】
请根据以下提供的多维度客户信息，对该客户的信用风险进行深度剖析，并最终预测其在美团借款后是否会发生逾期行为。

【客户多维数据】
1. 基本身份信息：
   - 基础画像：当前日期：2025-08-31。年龄：25。性别：女。常住地：河北省唐山市路北区。出生地：未知。户籍地：未知。户籍地址：未知。关联企业信息：。微信昵称：星星🌟。微信昵称获取日期：2023-07-26，微信昵称末次获...

【Assistant】:
{
  "customer_profile": "客户为25岁女性，常住河北省唐山市路北区，教育背景为河北东方学院，2021年4月实名认证美团账号，微信昵称近期更新。客户未婚，无房贷车贷记录，配偶有85690元负债，月还款10811元。美团内部账户显示无逾期记录，但配偶微贷逾期可能带来风险传染。客户生活轨迹稳定，常用外卖地址和WIFI地址集中在唐山市内，消费水平中等，有退款记录，但无支付失败记录。近期安装多款APP，显示出一定的消费和娱乐偏好。",
  "risk_analysis": {
    "负债结构": "客户自身无房贷车贷，但配偶有85690元负债，月还款10811元，存在一定的家庭债务压力。",
    "还款能力": "客户美团账户无逾期记录，但配偶微贷逾期可能影响家庭整体还款能力。",
    "逾期风险": "客户自身无逾期记录，但配偶逾期行为具有风险传染性，需警惕。",
    "多头查询": "客户近期无多头查询记录，风险较低。"
  },
  "demand_analysis": "客户有稳定的生活轨迹和中等消费水平，近期搜索关键词显示对餐饮、娱乐和生活服务有需求，但无明显的异常消费行为。",
  "reasoning_process": "客户自身信用记录良好，但配偶的微贷逾期行为增加了家庭债务风险。客户生活轨迹稳定，消费水平中等，无异常消费行为。尽管客户美团账户无逾期记录，但配偶的逾期行为具有风险传染性，需综合考虑家庭整体还款能力。基于客户近期行为稳定，且无明显的逾期前兆，评估其风险可控。",
  "cr

In [14]:
# 读取蒸馏数据
if not os.path.exists(output_jsonl):
    print("❌ 请先运行阶段1生成蒸馏数据")
else:
    # 读取 JSONL
    data = []
    with open(output_jsonl, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    
    print(f"✅ 加载 {len(data)} 条训练数据")
    
    # 转换为 Dataset
    dataset_dict = {
        'messages': [item['messages'] for item in data]
    }
    dataset = Dataset.from_dict(dataset_dict)
    
    # 划分训练集和验证集
    split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
    train_dataset = split_dataset['train']
    eval_dataset = split_dataset['test']
    
    print(f"📊 训练集: {len(train_dataset)} 条")
    print(f"📊 验证集: {len(eval_dataset)} 条")

✅ 加载 50 条训练数据
📊 训练集: 47 条
📊 验证集: 3 条


In [ ]:
# print(f"📥 加载 Tokenizer: {CONFIG['base_model']}")
# tokenizer = AutoTokenizer.from_pretrained(CONFIG['base_model'], trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"

# print(f"📥 加载模型: {CONFIG['base_model']}")
# model = AutoModelForCausalLM.from_pretrained(
#     CONFIG['base_model'],
#     device_map="auto",
#     torch_dtype=torch.bfloat16,
# )
# model.config.use_cache = False

# print("✅ 模型加载完成")

import time
print(f"📥 开始加载 Tokenizer: {CONFIG['base_model']}")
start_time = time.time()

tokenizer = AutoTokenizer.from_pretrained(CONFIG['base_model'], trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print(f"✅ Tokenizer 加载完成 (耗时: {time.time()-start_time:.1f}s)")

print(f"\n📥 开始加载模型,这可能需要 1-2 分钟...")
# print(f"💡 提示: 7B 模型需要约 14GB 显存")
start_time = time.time()

model = AutoModelForCausalLM.from_pretrained(
    CONFIG['base_model'],
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
print(f"✅ 模型加载完成 (耗时: {time.time()-start_time:.1f}s)")

model.config.use_cache = False

# 显示显存占用情况
print(f"\n📊 GPU 显存占用:")
print(f"  • GPU 0: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"  • GPU 1: {torch.cuda.memory_allocated(1)/1e9:.2f} GB")
print("\n✅ 所有组件加载完成,可以开始训练!")

📥 开始加载 Tokenizer: /home/sankuai/dolphinfs_chixinning/huggingface.co/Qwen/Qwen2.5-7B-Instruct
✅ Tokenizer 加载完成 (耗时: 0.4s)

📥 开始加载模型,这可能需要 1-2 分钟...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
print("⚙️ 配置 LoRA...")

peft_config = LoraConfig(
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    r=CONFIG['lora_r'],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['base_model'],
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model.enable_input_require_grads() 


model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)
model.config.pad_token_id = tokenizer.pad_token_id

# 打印可训练参数
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"📊 可训练参数: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")

In [ ]:
# 测试样本
test_messages = [
    {"role": "system", "content": "你是一个专业的信贷风控决策引擎。你的任务是根据用户的输入特征，进行严谨的逻辑推理，并严格按照 JSON 格式输出决策结果。决不能编造数据。"},
    {"role": "user", "content": "请评估该客户：\n【结构化特征】：授信额度=50000元, 当前余额=12000元, 定价利率=0.1800, 当前最大逾期天数=0, 历史最大逾期天数=0\n【非结构化特征】：人行征信良好，无逾期记录"}
]

# 生成 prompt
test_prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)

# 推理
model.eval()
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("📝 训练前模型输出:")
print(response)

In [ ]:
import transformers.loss.loss_utils as loss_utils

_original_fixed_cross_entropy = loss_utils.fixed_cross_entropy

def patched_fixed_cross_entropy(source, target, num_items_in_batch=None, ignore_index=-100, **kwargs):
    if num_items_in_batch is not None and isinstance(num_items_in_batch, torch.Tensor):
        num_items_in_batch = num_items_in_batch.to(source.device)
    return _original_fixed_cross_entropy(source, target, num_items_in_batch, ignore_index, **kwargs)

loss_utils.fixed_cross_entropy = patched_fixed_cross_entropy

In [ ]:
# 输出目录
output_dir = f"{CONFIG['output_dir']}_{timestamp}"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    num_train_epochs=CONFIG['num_epochs'],
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=10,
    bf16=True,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    gradient_checkpointing=True,
)

print(f"✅ 训练参数配置完成")
print(f"输出目录: {output_dir}")

# 开始训练

In [ ]:
# 格式化函数
def formatting_func(example):
    return tokenizer.apply_chat_template(example['messages'], tokenize=False)

# 创建 Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=formatting_func,
    tokenizer=tokenizer,
    max_seq_length=CONFIG['max_seq_length'],
    peft_config=peft_config,
)

print("🚀 开始训练...")
trainer.train()

print("✅ 训练完成!")

In [ ]:
# 保存最终模型
final_model_path = os.path.join(output_dir, "final_model")
print(f"💾 保存模型到: {final_model_path}")

trainer.model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("✅ 模型保存完成!")

In [ ]:
print('kernel test')

# 模型评估

In [ ]:
print(f"📥 加载微调后的模型: {final_model_path}")

# 释放训练时占用的显存
import gc
# del model
# del trainer
gc.collect()
torch.cuda.empty_cache()
print("🧹 已释放训练模型显存")

# 加载微调后的模型（先加载基础模型，再叠加 LoRA，最后合并）
from peft import PeftModel

# 1. 先加载完整的基础模型（确保所有层都有实际数据，避免 meta tensor 问题）
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG['base_model'],
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# 2. 加载 LoRA 适配器到基础模型上
finetuned_model = PeftModel.from_pretrained(base_model, final_model_path)

# 3. 合并 LoRA 权重到基础模型
finetuned_model = finetuned_model.merge_and_unload()

# 4. 清理临时变量
del base_model
gc.collect()
torch.cuda.empty_cache()

print("✅ 模型加载完成（LoRA 已合并）")

In [ ]:
# 推理
finetuned_model.eval()

# 改用 next(finetuned_model.parameters()).device
# merge_and_unload() 后 model.device 可能不准确
first_device = next(finetuned_model.parameters()).device
inputs = tokenizer(test_prompt, return_tensors="pt").to(first_device)

with torch.no_grad():
    outputs = finetuned_model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,   # 新增: 避免生成警告
        eos_token_id=tokenizer.eos_token_id,    # 新增: 明确结束条件
    )

# 只解码新生成的 token
generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
finetuned_response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# 调试: 如果输出为空，打印含 special tokens 的原始输出
if not finetuned_response:
    print("⚠️ 模型输出为空，尝试输出完整 token（含 special tokens）:")
    print(tokenizer.decode(generated_ids, skip_special_tokens=False))

print("="*60)
print("📝 训练后模型输出:")
print("="*60)
print(finetuned_response)
print("="*60)
print("\n📝 训练前模型输出:")
print("="*60)
print(response)